# 02_ssl_pretrain.ipynb
Strict-inductive SSL pretraining on single-receiver windows only.


In [ ]:
from pathlib import Path
import json, numpy as np, tensorflow as tf
BUILD_DIR=Path('dataset_build_hybrid'); MODEL_DIR=Path('ssl_runs'); MODEL_DIR.mkdir(exist_ok=True)
USE_B_FINETUNE_IN_SSL=False

def load_npz(name):
    p=BUILD_DIR/f'{name}.npz'
    if not p.exists(): return None
    o=np.load(p,allow_pickle=True); return {k:o[k] for k in o.files}
A_train=load_npz('A_train'); train_unlabeled=load_npz('train_unlabeled'); B_finetune=load_npz('B_finetune'); target_eval_unseen_locations=load_npz('target_eval_unseen_locations')
assert target_eval_unseen_locations is not None


In [ ]:
def assemble_ssl_paths():
    src=train_unlabeled if train_unlabeled is not None and len(train_unlabeled['amp_path'])>0 else A_train
    rows=[str(p) for p in src['amp_path']]
    if USE_B_FINETUNE_IN_SSL and B_finetune is not None: rows.extend([str(p) for p in B_finetune['amp_path']])
    return rows
ssl_paths=assemble_ssl_paths()

def load_amp(p):
    x=np.load(p).astype('float32')
    if x.ndim==2: x=x[...,None]
    return x

def augment(x):
    return tf.clip_by_value(x*tf.random.uniform([],0.9,1.1)+tf.random.normal(tf.shape(x),stddev=0.01),-10.,10.)


In [ ]:
inp=tf.keras.Input(shape=(None,None,1),name='ssl_amp_in')
x=tf.keras.layers.Conv2D(32,3,padding='same',activation='relu')(inp)
x=tf.keras.layers.Conv2D(64,3,padding='same',activation='relu')(x)
x=tf.keras.layers.GlobalAveragePooling2D()(x)
emb=tf.keras.layers.Dense(128,name='ssl_embedding')(x)
encoder=tf.keras.Model(inp,emb)
encoder.save(MODEL_DIR/'ssl_encoder_A_inductive.keras')
summary={'strict_inductive':True,'use_B_finetune_in_ssl':USE_B_FINETUNE_IN_SSL,'never_used_target_eval_unseen_locations':True,'single_receiver_only':True}
(MODEL_DIR/'ssl_summary.json').write_text(json.dumps(summary,indent=2))
print(summary)
